In [2]:
import random

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE, BatchNorm
from torch_geometric.utils import dropout_adj, from_scipy_sparse_matrix
from torch_geometric.data import Data
from torch.utils.data import DataLoader, TensorDataset
from torch.nn import BatchNorm1d

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import NMF
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

import scipy.sparse as sp

import networkx as nx
import matplotlib.pyplot as plt

from tqdm import tqdm

In [83]:
# Load the data
#data = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/NORMALIZED_EXP/normalized_weight_matrix_all_GRN_100_01_0.2filter.csv", index_col=0)
data = pd.read_csv("/Users/nirwantandukar/input_data/DREAM5/normalized_weight_matrix_DREAM5_net1_0.2filter.csv", index_col=0)
print(f"Loaded data with shape: {data.shape}")

# Get the list of algorithms
columns = data.columns
algorithms = sorted(set(col.split('.')[0] for col in columns))
print(f"Algorithms found: {algorithms}")

num_genes = data.shape[0]
gene_names = data.index.tolist()
print(f"Number of genes: {num_genes}")

# Node features (identity matrix)
node_features = np.identity(num_genes)
print(f"Node features shape: {node_features.shape}")

Loaded data with shape: (1643, 18073)
Algorithms found: ['GRN_ARACNE', 'GRN_C3NET', 'GRN_CLR', 'GRN_GENIE3_ET', 'GRN_GENIE3_RF', 'GRN_GRNBOOST2', 'GRN_KBOOST', 'GRN_LEAP', 'GRN_MRNET', 'GRN_MRNETB', 'GRN_PCIT']
Number of genes: 1643
Node features shape: (1643, 1643)


In [535]:
# Get algorithm list (already sorted)
algorithms = sorted(set(col.split('.')[0] for col in data.columns))

# Remove by index (Python is 0-based, so 6th = index 5, 10th = index 9)
algos_to_remove = [algorithms[15], algorithms[16]]
print(f"Removing: {algos_to_remove}")

# Remove columns that belong to these algorithms
columns_to_keep = [col for col in data.columns if not any(col.startswith(algo + '.') for algo in algos_to_remove)]

# Filter the data
data_filtered = data[columns_to_keep]

# Update algorithms list
algorithms_filtered = [algo for algo in algorithms if algo not in algos_to_remove]

# Confirm
print(f"Filtered data shape: {data_filtered.shape}")
print(f"Remaining algorithms: {algorithms_filtered}")


Removing: ['GRN_PCIT', 'GRN_TIGRESS']
Filtered data shape: (100, 1500)
Remaining algorithms: ['GRN_ARACNE', 'GRN_BC3NET', 'GRN_C3NET', 'GRN_CLR', 'GRN_GENIE3_ET', 'GRN_GENIE3_RF', 'GRN_GRENIST', 'GRN_GRNBOOST2', 'GRN_INFERELATOR', 'GRN_KBOOST', 'GRN_LEAP', 'GRN_MEOMI', 'GRN_MRNET', 'GRN_MRNETB', 'GRN_NONLINEARODES']


In [6]:
# Functions

# Normalized_laplacian
def normalize_adj(adj):
    """Compute the normalized Laplacian."""
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    identity = sp.eye(adj.shape[0])  # Identity matrix of the same size as adj
    laplacian = identity - d_mat_inv_sqrt.dot(adj).dot(d_mat_inv_sqrt)  # Normalized Laplacian
    return laplacian.tocoo()

# GCN encoder
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=4, dropout_rate=0.01, edge_dropout_rate=0.01):
        super(GCNEncoder, self).__init__()
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        self.edge_dropout_rate = edge_dropout_rate
        
        # First layer: input to 2x hidden size
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, 2 * hidden_channels, cached=False))
        
        # Add intermediate layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(2 * hidden_channels, 2 * hidden_channels, cached=False))
        
        # Final layer: hidden size to output size
        self.convs.append(GCNConv(2 * hidden_channels, out_channels, cached=False))
    
        # Batch normalization layers for each convolution layer
        self.bns = torch.nn.ModuleList([BatchNorm1d(2 * hidden_channels, momentum=0.1) for _ in range(num_layers - 1)])

    
        # Dropout rate
        self.dropout = torch.nn.Dropout(p=dropout_rate)
    
        # LeakyReLU instead of ReLU
        self.activation_hidden = torch.nn.ReLU  # For hidden layers
        self.activation_output = torch.nn.Sigmoid()  # For final layer


    
    def forward(self, x, edge_index):
        # Apply edge dropout
        edge_index, _ = dropout_adj(edge_index, p=self.edge_dropout_rate, training=self.training)
        
        # First layer (without batch normalization)
        x = self.convs[0](x, edge_index)
        #x = self.activation_hidden(x)
        x = F.relu(x)
    
    # Intermediate layers with batch normalization and dropout
        for i in range(1, self.num_layers - 1):
            x_residual = x # For residual connection
            x = self.convs[i](x, edge_index)
            x = self.bns[i-1](x) # Apply batch normalization
            #x = self.activation_hidden(x)
            x = F.relu(x)
            x = self.dropout(x)
            
            # Add residual connection (skip connection)
            x = x + x_residual
    
        # Final layer (no activation or batch norm)
        x = self.convs[-1](x, edge_index)
        x = F.relu(x)
        #x = self.activation_output(x)
        return x


# GAE model decoder
def apply_gae_to_adjacency(adj_matrix, node_features, embedding_dim=16, epochs=200, dropout_rate=0.1, edge_dropout_rate=0.01, weight_decay=1e-8):
    # Convert adjacency matrix to sparse format
    adj_sparse = sp.csr_matrix(adj_matrix)
    # Normalize the adjacency matrix
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    # Get edge_index and edge_weight
    edge_index, edge_weight = from_scipy_sparse_matrix(adj_normalized)
    # Create data object
    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)

    # Define the GCNEncoder as the encoder for the GAE model
    encoder = GCNEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim, dropout_rate=dropout_rate, edge_dropout_rate=edge_dropout_rate)
    
    # Instantiate the GAE model by passing the encoder
    model = GAE(encoder)

    # Training settings
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)

    # Training loop
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        z = model.encode(data.x, data.edge_index)
        loss = model.recon_loss(z, data.edge_index)
        loss.backward()
        optimizer.step()

    # After training, reconstruct adjacency matrix
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index)
        adj_reconstructed = model.decoder.forward_all(z).cpu().numpy()
        
    # Normalize the reconstructed adjacency matrix to be between 0 and 1
    adj_min = adj_reconstructed.min()
    adj_max = adj_reconstructed.max()
    normalized_adj = (adj_reconstructed - adj_min) / (adj_max - adj_min)
    
    return normalized_adj

# Running GCN-GAE multiple times
def run_gae_with_algo_filtering(data, algorithms, thresholds, num_iterations=2, algo_threshold=2):
    """
    Run GAE multiple times per algorithm, apply filtering across iterations per algorithm,
    and return only the filtered binary adjacency matrices per algorithm.
    
    Parameters:
        data (pd.DataFrame): full input matrix (shape: num_genes × num_algorithms × genes)
        algorithms (list): list of algorithm prefixes like "GRN_PCIT"
        thresholds (list): list of float binarization thresholds (e.g. [0.4])
        num_iterations (int): number of repeated GAE runs per algorithm
        algo_threshold (int): number of iterations an edge must appear in to be kept
        
    Returns:
        filtered_results (dict): shape: threshold → algorithm → 100×100 filtered matrix
    """
    num_genes = data.shape[0]
    filtered_results = {}

    for threshold in thresholds:
        print(f"\n🔹 Running GAE with binarization threshold: {threshold}")
        filtered_results[f'threshold_{threshold}'] = {}

        for alg in algorithms:
            print(f"  🔸 Algorithm: {alg}")
            alg_columns = [col for col in data.columns if col.startswith(alg + '.')]

            if len(alg_columns) != num_genes:
                print(f"    ⚠️ Skipping {alg}: expected {num_genes} columns, got {len(alg_columns)}")
                continue  # Skip malformed input

            alg_matrix = data[alg_columns].values.reshape((num_genes, num_genes))
            matrices = []

            for iteration in range(num_iterations):
                print(f"    ⏳ Iteration {iteration + 1}/{num_iterations}")
                adj_reconstructed = apply_gae_to_adjacency(alg_matrix, node_features)
                adj_binary = (adj_reconstructed > threshold).astype(int)
                matrices.append(adj_binary)

            # Stack and filter: only keep edges seen in ≥ algo_threshold iterations
            matrices = np.stack(matrices, axis=0)  # shape: (num_iterations, 100, 100)
            filtered = (np.sum(matrices, axis=0) >= algo_threshold).astype(int)

            filtered_results[f'threshold_{threshold}'][alg] = filtered

    return filtered_results


In [78]:
# Define thresholds and number of iterations
thresholds = [0.2]  # Different thresholds to run
num_iterations = 10  # Number of runs for each threshold
algo_threshold=4 # how many algorithm 

# Run the GAE process and store results for each threshold and algorithm
filtered_results = run_gae_with_algo_filtering(
    data=data,
    algorithms=algorithms,
    thresholds=thresholds,
    num_iterations=num_iterations,
    algo_threshold=algo_threshold
)




🔹 Running GAE with binarization threshold: 0.2
  🔸 Algorithm: GRN_ARACNE
    ⏳ Iteration 1/10


/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


    ⏳ Iteration 2/10
    ⏳ Iteration 3/10
    ⏳ Iteration 4/10
    ⏳ Iteration 5/10
    ⏳ Iteration 6/10
    ⏳ Iteration 7/10
    ⏳ Iteration 8/10
    ⏳ Iteration 9/10
    ⏳ Iteration 10/10
  🔸 Algorithm: GRN_C3NET
    ⏳ Iteration 1/10


/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


    ⏳ Iteration 2/10
    ⏳ Iteration 3/10
    ⏳ Iteration 4/10
    ⏳ Iteration 5/10
    ⏳ Iteration 6/10
    ⏳ Iteration 7/10
    ⏳ Iteration 8/10
    ⏳ Iteration 9/10
    ⏳ Iteration 10/10
  🔸 Algorithm: GRN_CLR
    ⏳ Iteration 1/10


/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


... (13 outputs left out)

/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


    ⏳ Iteration 2/10
    ⏳ Iteration 3/10
    ⏳ Iteration 4/10
    ⏳ Iteration 5/10
    ⏳ Iteration 6/10
    ⏳ Iteration 7/10
    ⏳ Iteration 8/10
    ⏳ Iteration 9/10
    ⏳ Iteration 10/10
  🔸 Algorithm: GRN_PCIT
    ⏳ Iteration 1/10


/opt/anaconda3/envs/machine_learning/lib/python3.12/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'dropout_adj' is deprecated, use 'dropout_edge' instead
  warnings.warn(out)


    ⏳ Iteration 2/10
    ⏳ Iteration 3/10
    ⏳ Iteration 4/10
    ⏳ Iteration 5/10
    ⏳ Iteration 6/10
    ⏳ Iteration 7/10
    ⏳ Iteration 8/10
    ⏳ Iteration 9/10
    ⏳ Iteration 10/10


In [10]:
# Applying NMF to 17 algorithm
def fuse_with_joint_nmf_from_filtered(filtered_results, threshold, rank=20, num_genes=1643):
    """
    Fuses multiple filtered binary adjacency matrices using joint NMF.
    Each matrix must be (num_genes x num_genes), one per algorithm.
    """
    all_flattened = []

    for alg, matrix in filtered_results[f'threshold_{threshold}'].items():
        assert matrix.shape == (num_genes, num_genes), f"{alg} shape is invalid: {matrix.shape}"
        flattened = matrix.flatten()
        all_flattened.append(flattened)

    # all_flattened = 17 x 10000 → each row is one algorithm's flattened adjacency matrix
    nmf_input = np.array(all_flattened)  # shape: (17, 10000)
    print(f"Matrix shape (before transpose): {nmf_input.shape}")

    V = nmf_input.T  # (10000, 17)
    print(f"NMF input shape: {V.shape}")

    nmf_model = NMF(n_components=rank, init='random', random_state=42, max_iter=1000)
    W = nmf_model.fit_transform(V)
    H = nmf_model.components_

    # Reconstruct the fused graph
    V_reconstructed = np.dot(W, H).mean(axis=1)  # shape: (10000,)

    # RESHAPE using known num_genes = 100, not inferred!
    fused_adj = V_reconstructed.reshape((num_genes, num_genes))

    # Normalize to [0, 1]
    fused_adj = (fused_adj - fused_adj.min()) / (fused_adj.max() - fused_adj.min())

    return fused_adj



In [12]:
# Running the NMF fusion
fused_adj_nmf = fuse_with_joint_nmf_from_filtered(filtered_results, threshold=0, rank=20)


Matrix shape (before transpose): (11, 2699449)
NMF input shape: (2699449, 11)


In [13]:
fused_adj_nmf.shape

(1643, 1643)

In [16]:
threshold = 0.6
fused_binary = (fused_adj_nmf >= threshold).astype(int)

In [18]:
fused_binary

array([[1, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 0, 1, 0],
       [0, 0, 0, ..., 0, 0, 1]])

In [20]:
# Function to extract nodes and edges from the binary matrix
def count_nodes_edges(final_matrix):
    # Count edges in the upper triangle (to avoid double counting for undirected graphs)
    num_edges = np.sum(np.triu(final_matrix, k=1))
    
    # Count nodes (genes with at least one edge, i.e., rows/columns with non-zero values)
    connected_nodes = np.sum(np.any(final_matrix, axis=1))  # Count rows with non-zero values
    num_nodes = connected_nodes
    
    return num_nodes, num_edges

# Function to extract gene interactions (edges) from a binarized matrix
def extract_gene_interactions(final_matrix, gene_names):
    edges = []
    num_genes = final_matrix.shape[0]

    # Loop through the upper triangle of the matrix to avoid duplicates (since it's undirected)
    for i in range(num_genes):
        for j in range(i + 1, num_genes):
            if final_matrix[i, j] == 1:  # Check if there's an interaction
                edges.append(f"{gene_names[i]}-{gene_names[j]}")
    
    return edges


In [22]:
# Count nodes and edges
num_nodes, num_edges = count_nodes_edges(fused_binary)
print(f"Nodes: {num_nodes}, Edges: {num_edges}")

# Get the edge list
nmf_edge_list = extract_gene_interactions(fused_binary, gene_names)
print(f"Sample edges: {nmf_edge_list[:5]}")

Nodes: 1642, Edges: 3827
Sample edges: ['G1-G1130', 'G10-G1048', 'G10-G121', 'G10-G1215', 'G10-G1235']


In [32]:
# Convert ground truth matrix to edge list
def convert_matrix_to_edge_list(matrix):
    edges = []
    for source in matrix.index:
        for target in matrix.columns:
            if matrix.at[source, target] == 1:
                edges.append((source, target))
    return edges

# Load known (ground truth) interactions
file_path = "/Users/nirwantandukar/input_data/Database.DREAM4/GS/dream4_100_01_gs.csv"
#file_path = "/Users/nirwantandukar/input_data/DREAM5/net1_in_silico_gold_standard.csv"
ground_truth = pd.read_csv(file_path, index_col=0)
print(f"Ground truth loaded with shape: {ground_truth.shape}")

# Get the ground truth edges
known_edges_set = set(convert_matrix_to_edge_list(ground_truth))
print(f"Number of known edges: {len(known_edges_set)}")
known_edges_set

Ground truth loaded with shape: (100, 100)
Number of known edges: 175


{('G1', 'G3'),
 ('G1', 'G4'),
 ('G10', 'G37'),
 ('G10', 'G44'),
 ('G10', 'G50'),
 ('G100', 'G10'),
 ('G100', 'G88'),
 ('G15', 'G14'),
 ('G23', 'G24'),
 ('G25', 'G7'),
 ('G26', 'G27'),
 ('G26', 'G28'),
 ('G26', 'G29'),
 ('G26', 'G30'),
 ('G26', 'G31'),
 ('G26', 'G32'),
 ('G26', 'G33'),
 ('G26', 'G34'),
 ('G26', 'G35'),
 ('G36', 'G37'),
 ('G37', 'G1'),
 ('G37', 'G10'),
 ('G37', 'G23'),
 ('G37', 'G24'),
 ('G37', 'G25'),
 ('G37', 'G29'),
 ('G37', 'G47'),
 ('G37', 'G48'),
 ('G37', 'G51'),
 ('G37', 'G59'),
 ('G37', 'G60'),
 ('G37', 'G61'),
 ('G37', 'G62'),
 ('G37', 'G64'),
 ('G37', 'G67'),
 ('G37', 'G68'),
 ('G37', 'G69'),
 ('G37', 'G70'),
 ('G37', 'G71'),
 ('G37', 'G72'),
 ('G37', 'G73'),
 ('G37', 'G74'),
 ('G37', 'G75'),
 ('G37', 'G8'),
 ('G38', 'G39'),
 ('G38', 'G9'),
 ('G40', 'G41'),
 ('G42', 'G1'),
 ('G43', 'G44'),
 ('G43', 'G45'),
 ('G44', 'G10'),
 ('G44', 'G31'),
 ('G44', 'G50'),
 ('G44', 'G62'),
 ('G44', 'G63'),
 ('G44', 'G64'),
 ('G45', 'G44'),
 ('G46', 'G10'),
 ('G46', 'G16'),
 ('G

In [40]:
### DREAM 5

# Load the edge list CSV
file_path = "/Users/nirwantandukar/input_data/DREAM5/net1_in_silico_gold_standard.csv"
ground_truth = pd.read_csv(file_path)

# Confirm expected columns are there
print("Columns:", ground_truth.columns)

# Convert edge list to set of tuples
known_edges_set = set(zip(ground_truth['from'], ground_truth['to']))
print(f"Loaded {len(known_edges_set)} known edges from edge list.")
known_edges_set

Columns: Index(['from', 'to', 'weight'], dtype='object')
Loaded 4012 known edges from edge list.


{('G82', 'G230'),
 ('G136', 'G746'),
 ('G171', 'G951'),
 ('G187', 'G686'),
 ('G68', 'G845'),
 ('G120', 'G1182'),
 ('G154', 'G1155'),
 ('G110', 'G727'),
 ('G78', 'G511'),
 ('G176', 'G657'),
 ('G115', 'G754'),
 ('G141', 'G1310'),
 ('G120', 'G181'),
 ('G120', 'G1133'),
 ('G88', 'G80'),
 ('G52', 'G1297'),
 ('G84', 'G1626'),
 ('G78', 'G115'),
 ('G165', 'G1206'),
 ('G136', 'G1300'),
 ('G109', 'G946'),
 ('G63', 'G403'),
 ('G84', 'G590'),
 ('G188', 'G938'),
 ('G50', 'G1466'),
 ('G144', 'G1586'),
 ('G100', 'G645'),
 ('G136', 'G1114'),
 ('G141', 'G291'),
 ('G136', 'G1272'),
 ('G171', 'G1461'),
 ('G120', 'G294'),
 ('G28', 'G1250'),
 ('G126', 'G1266'),
 ('G194', 'G960'),
 ('G136', 'G339'),
 ('G59', 'G230'),
 ('G121', 'G523'),
 ('G120', 'G555'),
 ('G133', 'G893'),
 ('G141', 'G1315'),
 ('G122', 'G758'),
 ('G84', 'G1080'),
 ('G120', 'G251'),
 ('G26', 'G1299'),
 ('G171', 'G832'),
 ('G120', 'G496'),
 ('G120', 'G122'),
 ('G133', 'G410'),
 ('G169', 'G675'),
 ('G141', 'G679'),
 ('G6', 'G1043'),
 ('G84', '

In [54]:
def convert_str_edges_to_tuples(edge_list):
    edge_tuples = set()
    for edge in edge_list:
        a, b = edge.split('-')
        # Add both (a, b) and (b, a) to ensure directionality doesn't affect matching
        edge_tuples.add((a, b))
        edge_tuples.add((b, a))
    return edge_tuples

predicted_edge_set = convert_str_edges_to_tuples(nmf_edge_list)

def symmetrize_edges(edge_set):
    symmetric_set = set()
    for a, b in edge_set:
        symmetric_set.add((a, b))
        symmetric_set.add((b, a))
    return symmetric_set

known_edges_symmetric = symmetrize_edges(known_edges_set)


{('G187', 'G686'),
 ('G110', 'G727'),
 ('G115', 'G754'),
 ('G1342', 'G121'),
 ('G84', 'G1626'),
 ('G78', 'G115'),
 ('G165', 'G1206'),
 ('G511', 'G105'),
 ('G126', 'G31'),
 ('G100', 'G645'),
 ('G141', 'G291'),
 ('G1033', 'G141'),
 ('G194', 'G960'),
 ('G136', 'G339'),
 ('G121', 'G523'),
 ('G133', 'G893'),
 ('G122', 'G758'),
 ('G1631', 'G84'),
 ('G392', 'G52'),
 ('G657', 'G84'),
 ('G26', 'G1299'),
 ('G621', 'G125'),
 ('G133', 'G410'),
 ('G84', 'G1244'),
 ('G624', 'G45'),
 ('G84', 'G424'),
 ('G1616', 'G78'),
 ('G1609', 'G16'),
 ('G754', 'G119'),
 ('G1276', 'G63'),
 ('G58', 'G359'),
 ('G16', 'G1368'),
 ('G100', 'G1524'),
 ('G141', 'G828'),
 ('G625', 'G129'),
 ('G1397', 'G99'),
 ('G538', 'G115'),
 ('G1363', 'G194'),
 ('G1249', 'G192'),
 ('G1269', 'G68'),
 ('G39', 'G1262'),
 ('G350', 'G84'),
 ('G115', 'G1375'),
 ('G84', 'G1172'),
 ('G120', 'G1523'),
 ('G16', 'G1378'),
 ('G988', 'G169'),
 ('G12', 'G244'),
 ('G137', 'G755'),
 ('G171', 'G791'),
 ('G136', 'G621'),
 ('G886', 'G110'),
 ('G1555', 'G

In [56]:
# True Positives (correct predictions)
true_positives = predicted_edge_set.intersection(known_edges_symmetric)

# False Positives (predicted but not known)
false_positives = predicted_edge_set - known_edges_symmetric

# False Negatives (missed known edges)
false_negatives = known_edges_symmetric - predicted_edge_set

# Precision, Recall, F1
tp = len(true_positives) // 2  # divide by 2 because of symmetric edges
fp = len(false_positives) // 2
fn = len(false_negatives) // 2

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"TP: {tp}, FP: {fp}, FN: {fn}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}")
predicted_edge_set

TP: 384, FP: 3443, FN: 3612
Precision: 0.1003, Recall: 0.0961, F1 Score: 0.0982


{('G187', 'G686'),
 ('G80', 'G831'),
 ('G1277', 'G609'),
 ('G1349', 'G493'),
 ('G1048', 'G695'),
 ('G590', 'G534'),
 ('G1506', 'G1233'),
 ('G1246', 'G725'),
 ('G218', 'G96'),
 ('G890', 'G1349'),
 ('G1592', 'G189'),
 ('G1324', 'G999'),
 ('G819', 'G1552'),
 ('G993', 'G226'),
 ('G1173', 'G1318'),
 ('G136', 'G339'),
 ('G1088', 'G1120'),
 ('G1378', 'G687'),
 ('G890', 'G736'),
 ('G753', 'G1329'),
 ('G121', 'G890'),
 ('G590', 'G1284'),
 ('G181', 'G1589'),
 ('G1229', 'G1380'),
 ('G26', 'G1299'),
 ('G939', 'G1235'),
 ('G534', 'G493'),
 ('G1604', 'G277'),
 ('G493', 'G1291'),
 ('G782', 'G939'),
 ('G84', 'G1244'),
 ('G1215', 'G131'),
 ('G1025', 'G426'),
 ('G426', 'G1324'),
 ('G1329', 'G1074'),
 ('G1360', 'G407'),
 ('G1098', 'G689'),
 ('G386', 'G819'),
 ('G515', 'G1330'),
 ('G973', 'G939'),
 ('G1402', 'G1558'),
 ('G263', 'G500'),
 ('G1431', 'G470'),
 ('G863', 'G1244'),
 ('G454', 'G973'),
 ('G1244', 'G1246'),
 ('G354', 'G1235'),
 ('G225', 'G1048'),
 ('G178', 'G1625'),
 ('G760', 'G973'),
 ('G131', 'G

In [50]:
#### AUPR and AUROC

from sklearn.metrics import roc_auc_score, average_precision_score

def compute_auroc_aupr(predicted_adj, ground_truth_df, gene_names):
    """
    Computes AUROC and AUPR given predicted adjacency (scores) and ground truth (binary).
    Assumes both are 100x100 matrices with matching gene order.
    """

    # Convert ground truth DataFrame to a numpy binary matrix
    gt_matrix = ground_truth_df.loc[gene_names, gene_names].values.astype(int)

    # Flatten only upper triangle (excluding diagonal)
    triu_indices = np.triu_indices_from(gt_matrix, k=1)
    gt_flat = gt_matrix[triu_indices]
    pred_flat = predicted_adj[triu_indices]

    # AUROC and AUPR
    auroc = roc_auc_score(gt_flat, pred_flat)
    aupr = average_precision_score(gt_flat, pred_flat)

    print(f"AUROC: {auroc:.4f}")
    print(f"AUPR : {aupr:.4f}")
    
    return auroc, aupr


In [52]:
# Call the scorer
compute_auroc_aupr(fused_adj_nmf, ground_truth, gene_names)

KeyError: "None of [Index(['G1', 'G10', 'G100', 'G1000', 'G1001', 'G1002', 'G1003', 'G1004',\n       'G1005', 'G1006',\n       ...\n       'G990', 'G991', 'G992', 'G993', 'G994', 'G995', 'G996', 'G997', 'G998',\n       'G999'],\n      dtype='object', length=1643)] are in the [index]"

In [754]:
# Call the scorer
compute_auroc_aupr(fused_adj_nmf, ground_truth, gene_names)

AUROC: 0.8113
AUPR : 0.0866


(0.811267863644896, 0.08661751251712864)

In [72]:
# DREAM 5 


from sklearn.metrics import precision_recall_curve, roc_curve, auc
import numpy as np
import pandas as pd

def compute_auroc_aupr(predicted_adj, ground_truth_df, gene_names):
    """
    Computes AUROC and AUPR from:
    - predicted_adj: N×N numpy array of scores
    - ground_truth_df: edge list with columns ['from', 'to']
    - gene_names: list of gene names matching predicted_adj
    """

    # Build a ground truth binary matrix from edge list
    N = len(gene_names)
    gene_to_index = {gene: i for i, gene in enumerate(gene_names)}
    gt_binary = np.zeros((N, N), dtype=int)

    for _, row in ground_truth_df.iterrows():
        src, tgt = row['from'], row['to']
        if src in gene_to_index and tgt in gene_to_index:
            i, j = gene_to_index[src], gene_to_index[tgt]
            gt_binary[i, j] = 1

    # Flatten upper triangle only (no self loops)
    triu = np.triu_indices(N, k=1)
    y_true = gt_binary[triu]
    y_scores = predicted_adj[triu]

    # Compute metrics
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    precision, recall, _ = precision_recall_curve(y_true, y_scores)

    auroc = auc(fpr, tpr)
    aupr = auc(recall, precision)

    print(f"AUROC: {auroc:.4f}")
    print(f"AUPR:  {aupr:.4f}")

    return auroc, aupr


import numpy as np

def edge_list_to_binary_matrix(edge_list, gene_names):
    N = len(gene_names)
    gene_to_index = {gene: i for i, gene in enumerate(gene_names)}
    matrix = np.zeros((N, N), dtype=int)

    for edge in edge_list:
        try:
            src, tgt = edge.split('-')
            if src in gene_to_index and tgt in gene_to_index:
                i, j = gene_to_index[src], gene_to_index[tgt]
                matrix[i, j] = 1
        except ValueError:
            continue  # skip malformed lines

    return matrix



In [76]:
# Convert edge list into binary predicted matrix
predicted_adj = edge_list_to_binary_matrix(nmf_edge_list, gene_names)

# Run scorer using your ground truth edge list
compute_auroc_aupr(predicted_adj, ground_truth, gene_names)


AUROC: 0.5463
AUPR:  0.0772


(0.5463003659104908, 0.07715714461547274)

In [70]:
nmf_edge_list

['G1-G1130',
 'G10-G1048',
 'G10-G121',
 'G10-G1215',
 'G10-G1235',
 'G10-G1246',
 'G10-G1284',
 'G10-G131',
 'G10-G1312',
 'G10-G1349',
 'G10-G1368',
 'G10-G1377',
 'G10-G1401',
 'G10-G1462',
 'G10-G1543',
 'G10-G1558',
 'G10-G1616',
 'G10-G1624',
 'G10-G1639',
 'G10-G176',
 'G10-G225',
 'G10-G228',
 'G10-G236',
 'G10-G272',
 'G10-G299',
 'G10-G34',
 'G10-G354',
 'G10-G367',
 'G10-G423',
 'G10-G427',
 'G10-G454',
 'G10-G493',
 'G10-G498',
 'G10-G534',
 'G10-G545',
 'G10-G569',
 'G10-G662',
 'G10-G695',
 'G10-G736',
 'G10-G760',
 'G10-G782',
 'G10-G812',
 'G10-G84',
 'G10-G863',
 'G10-G881',
 'G10-G939',
 'G10-G940',
 'G10-G973',
 'G100-G1102',
 'G100-G604',
 'G100-G673',
 'G1001-G1046',
 'G1001-G1098',
 'G1001-G1157',
 'G1001-G1520',
 'G1001-G1526',
 'G1001-G262',
 'G1001-G619',
 'G1001-G65',
 'G1001-G689',
 'G1001-G78',
 'G1002-G1037',
 'G1002-G1133',
 'G1002-G1270',
 'G1002-G1344',
 'G1002-G1436',
 'G1002-G171',
 'G1002-G347',
 'G1002-G389',
 'G1002-G409',
 'G1002-G635',
 'G1002-G63